In [1]:
!pip install tensorflow scikit-learn pandas

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
ss
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

Step 1 — Load and Explore Dataset

In [ ]:
df = pd.read_csv('fake_news.csv')  # change path

print(df.head())

# Check distribution
print(df['Label'].value_counts())

  Article_ID                 Title  \
0      A2000  muijjmRMcIbPCQpbtzKf   
1      A2001  VOjCRQQMXtCT nldGjIx   
2      A2002  QoGYMqVyCWOgpHcKIyHM   
3      A2003  kID scUjPxhietgoGRUD   
4      A2004  DegQqeyENSqgPsddDQQR   

                                             Content Label  
0  JDpKIOFLHtNwhzimEwjBKdNzCkpbH poZPpnMWdQSJSLiz...  Fake  
1  RiOJESKpHqMkeYWORXfOdWpyIVjejRaAmeohDWlCBPySpe...  Fake  
2  FkkkTcdOcCCyJcMIyYwTMlV kwHIiSWjBeHOiKUVVsm SS...  Fake  
3  hMNaVLnQOUMiPfHVlgXedmnIMxeePtyzXUFKFhFLrTqMjo...  Real  
4  xTeRubcTsRBK FqeZLdyUOLDffUFjYnCyRVd dYpsEGLZG...  Real  
Label
Real    276
Fake    224
Name: count, dtype: int64


Step 2 — Text Preprocessing (TF-IDF)

In [ ]:
X = df['Content']
y = df['Label']

# Convert categorical labels to numerical (0s and 1s)
y_encoded, unique_labels = pd.factorize(y)
y = y_encoded

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(X).toarray()

Step 3 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Step 4 — Build Neural Network

In [ ]:
model = Sequential()

# Input layer + Hidden layer
model.add(Dense(128, activation='relu', input_dim=X.shape[1]))

# Hidden layer
model.add(Dense(64, activation='relu'))

# Output layer
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Step 5 — Compile and Train

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.4600 - loss: 0.6929 - val_accuracy: 0.5500 - val_loss: 0.6918
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5550 - loss: 0.6716 - val_accuracy: 0.5600 - val_loss: 0.6897
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6100 - loss: 0.6353 - val_accuracy: 0.5600 - val_loss: 0.6906
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8625 - loss: 0.5539 - val_accuracy: 0.5600 - val_loss: 0.6950
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9950 - loss: 0.3978 - val_accuracy: 0.5300 - val_loss: 0.7032
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.2099 - val_accuracy: 0.4900 - val_loss: 0.7124
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.0816 - val_accuracy: 0.5000 - val_loss: 0.7238
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0291 - val_accuracy: 0.4800 - v

Step 6 — Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4700 - loss: 0.7524
Test Accuracy: 0.4699999988079071


Step 7 — Test with New Headlines

In [ ]:
def predict_news(text):
    vec = vectorizer.transform([text]).toarray()
    pred = model.predict(vec)

    # Corrected logic: if probability of 'Real' (class 1) is > 0.5, then it's Real News
    if pred[0][0] > 0.5:
        return "Real News"
    else:
        return "Fake News"


print(predict_news("Prime Minister meets with international leaders to discuss traden"))
print(predict_news("Aliens landed in New York and took control."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Real News
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Real News


# explanation

**Neural Network Architecture**: A sequential neural network was used with an input layer, two dense hidden layers (128 neurons, then 64 neurons), and a dense output layer.

**Epochs:** The model was trained for 15 epochs.
**Activation Functions:** The relu activation function was used for the hidden layers, and the sigmoid activation function was used for the output layer.

**Accuracy:** The model achieved a test accuracy of approximately 47%.
